# L39 - MDP Framing for Simulation Environments

**Learning objectives**
- Translate a simulation control problem into states, actions, rewards, and transitions.
- Compute Bellman backups on a small discrete MDP.
- Interpret the policy implied by value iteration.
- Connect queue-control intuition to the MDP formalism in Appendix D.

In [ ]:
import numpy as np
import pandas as pd

states = [0, 1, 2, 3]
actions = {0: 'hold staffing', 1: 'add extra server'}
gamma = 0.95

transitions = {
    (0, 0): [(1.0, 0, 0.0)],
    (0, 1): [(1.0, 0, -0.5)],
    (1, 0): [(0.3, 0, -1.0), (0.5, 1, -1.0), (0.2, 2, -1.0)],
    (1, 1): [(0.7, 0, -1.5), (0.3, 1, -1.5)],
    (2, 0): [(0.2, 1, -2.0), (0.5, 2, -2.0), (0.3, 3, -2.0)],
    (2, 1): [(0.5, 1, -2.5), (0.5, 2, -2.5)],
    (3, 0): [(0.2, 2, -4.0), (0.8, 3, -4.0)],
    (3, 1): [(0.6, 2, -4.5), (0.4, 3, -4.5)],
}

In [ ]:
def bellman_backup(state, action, values):
    total = 0.0
    for prob, next_state, reward in transitions[(state, action)]:
        total += prob * (reward + gamma * values[next_state])
    return total

def value_iteration(tol=1e-8, max_iter=500):
    values = {s: 0.0 for s in states}
    for _ in range(max_iter):
        new_values = {}
        for s in states:
            new_values[s] = max(bellman_backup(s, a, values) for a in actions)
        delta = max(abs(new_values[s] - values[s]) for s in states)
        values = new_values
        if delta < tol:
            break
    policy = {s: max(actions, key=lambda a: bellman_backup(s, a, values)) for s in states}
    return values, policy

values, policy = value_iteration()
pd.DataFrame({
    'state': states,
    'value': [values[s] for s in states],
    'best_action': [actions[policy[s]] for s in states],
})

## Interpretation

This tiny MDP is not a full queueing model. It is a decision sketch. The important point is that the state already encodes operational congestion, the action changes service capacity, and the reward penalizes both waiting and staffing cost. That is the conceptual bridge from simulation to RL.

In [ ]:
for s in states:
    q_values = {actions[a]: bellman_backup(s, a, values) for a in actions}
    print(f'state={s} -> {q_values}')

## Try It Yourself

1. Increase the staffing penalty for action `add extra server` and rerun value iteration.
2. Add a fifth state representing severe congestion.
3. Explain how the state and reward design would change for a clinic with multiple queues.